In [3]:
# ============================================================
# TRUE VLA: 4-STEP FLOW
# STEP 1: Show all 7 objects (real per-object crops, not a lookup table)
# STEP 2: Widget-based object selection (buttons, not free text)
# STEP 3: Train CNN + language model live, print epoch table AND a
#         feature -> score table for all 7 objects
# STEP 4: Live robot motion with continuous camera capture
# Google Colab ready
# ============================================================

# ── 1. INSTALL ────────────────────────────────────────────────
!pip install pybullet ipywidgets opencv-python torch -q

# ── 2. IMPORTS ───────────────────────────────────────────────
import time
import random
import numpy as np
import pybullet as p
import pybullet_data
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 64
TRAIN_EPOCHS = 10
BATCH_SIZE = 32

VOCAB = ["<pad>", "pick", "the", "a", "small", "big", "medium", "green", "red", "blue",
         "yellow", "box", "t-shirt", "bottle", "ball", "grab", "move", "get"]
WORD_TO_ID = {w: i for i, w in enumerate(VOCAB)}
MAX_LEN = 6

# ── 3. OBJECTS ───────────────────────────────────────────────
OBJECTS = [
    {"name": "Small Green Box", "item": "box", "color": "green", "size": "small",
     "rgb": (40, 175, 70), "pos": (0.55, 0.36, 0.045), "geom": "box",
     "half": (0.045, 0.045, 0.045), "mass": 0.20},
    {"name": "Big Green Box", "item": "box", "color": "green", "size": "big",
     "rgb": (40, 175, 70), "pos": (0.55, 0.20, 0.075), "geom": "box",
     "half": (0.075, 0.075, 0.075), "mass": 0.30},
    {"name": "Small Red T-Shirt", "item": "t-shirt", "color": "red", "size": "small",
     "rgb": (220, 45, 45), "pos": (0.55, 0.02, 0.025), "geom": "box",
     "half": (0.070, 0.045, 0.025), "mass": 0.15},
    {"name": "Big Red T-Shirt", "item": "t-shirt", "color": "red", "size": "big",
     "rgb": (220, 45, 45), "pos": (0.55, -0.18, 0.040), "geom": "box",
     "half": (0.105, 0.070, 0.040), "mass": 0.25},
    {"name": "Small Blue Bottle", "item": "bottle", "color": "blue", "size": "small",
     "rgb": (45, 90, 220), "pos": (0.55, -0.34, 0.060), "geom": "cylinder",
     "radius": 0.032, "height": 0.12, "mass": 0.15},
    {"name": "Big Blue Bottle", "item": "bottle", "color": "blue", "size": "big",
     "rgb": (45, 90, 220), "pos": (0.38, 0.30, 0.085), "geom": "cylinder",
     "radius": 0.045, "height": 0.17, "mass": 0.25},
    {"name": "Yellow Ball", "item": "ball", "color": "yellow", "size": "medium",
     "rgb": (235, 190, 35), "pos": (0.38, 0.05, 0.045), "geom": "sphere",
     "radius": 0.045, "mass": 0.12},
]

# ── 4. PYBULLET SETUP ─────────────────────────────────────────
try:
    p.disconnect()
except Exception:
    pass

p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.8)

ground_id = p.loadURDF("plane.urdf", useFixedBase=True)
street_visual = p.createVisualShape(p.GEOM_BOX, halfExtents=[1.5, 1.0, 0.002], rgbaColor=[0.35, 0.35, 0.35, 1.0])
p.createMultiBody(baseMass=0, baseVisualShapeIndex=street_visual, basePosition=[0.5, 0, 0.003])

kuka_id = p.loadURDF("kuka_iiwa/model.urdf", basePosition=[0, 0, 0], useFixedBase=True)
NUM_JOINTS = p.getNumJoints(kuka_id)
EE_INDEX = 6

ROBOT_COLOR = [0.45, 0.45, 0.50, 1.0]
p.changeVisualShape(kuka_id, -1, rgbaColor=ROBOT_COLOR)
for link_idx in range(NUM_JOINTS):
    p.changeVisualShape(kuka_id, link_idx, rgbaColor=ROBOT_COLOR)

REST_POSE = [0.006418, 0.413184, -0.011401, -1.589317, 0.005379, 1.137684, -0.006539]
for j, angle in enumerate(REST_POSE):
    p.resetJointState(kuka_id, j, angle)

# ── 5. CREATE / RESET OBJECTS ─────────────────────────────────
def create_objects():
    for obj in OBJECTS:
        rgba = [c / 255.0 for c in obj["rgb"]] + [1.0]
        if obj["geom"] == "box":
            coll = p.createCollisionShape(p.GEOM_BOX, halfExtents=obj["half"])
            vis = p.createVisualShape(p.GEOM_BOX, halfExtents=obj["half"], rgbaColor=rgba)
        elif obj["geom"] == "cylinder":
            coll = p.createCollisionShape(p.GEOM_CYLINDER, radius=obj["radius"], height=obj["height"])
            vis = p.createVisualShape(p.GEOM_CYLINDER, radius=obj["radius"], length=obj["height"], rgbaColor=rgba)
        else:
            coll = p.createCollisionShape(p.GEOM_SPHERE, radius=obj["radius"])
            vis = p.createVisualShape(p.GEOM_SPHERE, radius=obj["radius"], rgbaColor=rgba)

        body_id = p.createMultiBody(baseMass=obj["mass"], baseCollisionShapeIndex=coll,
                                    baseVisualShapeIndex=vis, basePosition=obj["pos"])
        obj["body_id"] = body_id
        p.changeDynamics(body_id, -1, lateralFriction=0.8, spinningFriction=0.5,
                         rollingFriction=0.5, restitution=0.05)

grasp_constraint = None

def reset_scene():
    global grasp_constraint
    if grasp_constraint is not None:
        try:
            p.removeConstraint(grasp_constraint)
        except Exception:
            pass
        grasp_constraint = None

    for j, angle in enumerate(REST_POSE):
        p.resetJointState(kuka_id, j, angle)

    for obj in OBJECTS:
        if "body_id" in obj:
            try:
                p.removeBody(obj["body_id"])
            except Exception:
                pass
            del obj["body_id"]

    create_objects()
    for _ in range(15):
        p.stepSimulation()

reset_scene()

# ── 6. CAMERA ─────────────────────────────────────────────────
FULL_W, FULL_H = 560, 400
VIEW_MATRIX = p.computeViewMatrix([1.15, -0.75, 0.55], [0.48, 0.0, 0.08], [0, 0, 1])
PROJ_MATRIX = p.computeProjectionMatrixFOV(55, FULL_W / FULL_H, 0.1, 3.0)

def capture_rgb(w=FULL_W, h=FULL_H):
    _, _, rgba, _, _ = p.getCameraImage(w, h, VIEW_MATRIX, PROJ_MATRIX, renderer=p.ER_TINY_RENDERER)
    return np.reshape(rgba, (h, w, 4))[:, :, :3].astype(np.uint8)

def rgb_to_png_bytes(rgb):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    _, png = cv2.imencode(".png", bgr)
    return png.tobytes()

def capture_png():
    return rgb_to_png_bytes(capture_rgb())

# ── 7. REAL PER-OBJECT CROPPING (projects 3D position -> pixel) ──
# This is the fix that makes the CNN actually see something different
# per object, instead of the same fixed rectangle every time.
def project_point(pos, view_matrix, proj_matrix, width, height):
    view = np.array(view_matrix, dtype=np.float64).reshape(4, 4, order="F")
    proj = np.array(proj_matrix, dtype=np.float64).reshape(4, 4, order="F")
    point = np.array([pos[0], pos[1], pos[2], 1.0])
    clip = proj @ (view @ point)
    ndc = clip[:3] / clip[3]
    px = (ndc[0] * 0.5 + 0.5) * width
    py = (1.0 - (ndc[1] * 0.5 + 0.5)) * height
    return px, py

def get_object_crop(obj, crop_half=75, out_size=IMG_SIZE, jitter=True):
    """Capture the full scene, then crop a window centered on THIS
    object's real projected pixel position. Every object therefore
    produces a genuinely different image, so the CNN has to read
    color/shape/size from actual pixels."""
    rgb = capture_rgb(FULL_W, FULL_H)
    px, py = project_point(obj["pos"], VIEW_MATRIX, PROJ_MATRIX, FULL_W, FULL_H)

    if jitter:
        px += random.uniform(-8, 8)
        py += random.uniform(-8, 8)

    px, py = int(px), int(py)
    x0, x1 = max(0, px - crop_half), min(FULL_W, px + crop_half)
    y0, y1 = max(0, py - crop_half), min(FULL_H, py + crop_half)
    crop = rgb[y0:y1, x0:x1]
    if crop.shape[0] < 4 or crop.shape[1] < 4:
        crop = rgb  # fallback, should not normally trigger
    crop = cv2.resize(crop, (out_size, out_size))

    if jitter:
        factor = random.uniform(0.85, 1.15)
        crop = np.clip(crop.astype(np.float32) * factor, 0, 255).astype(np.uint8)

    tensor = torch.from_numpy(crop).permute(2, 0, 1).float() / 255.0
    return tensor

def get_object_crop_png(obj):
    rgb = capture_rgb(FULL_W, FULL_H)
    px, py = project_point(obj["pos"], VIEW_MATRIX, PROJ_MATRIX, FULL_W, FULL_H)
    px, py = int(px), int(py)
    x0, x1 = max(0, px - 75), min(FULL_W, px + 75)
    y0, y1 = max(0, py - 75), min(FULL_H, py + 75)
    crop = rgb[y0:y1, x0:x1]
    return rgb_to_png_bytes(crop)

# Precompute each object's base crop ONCE. Positions are static, so the
# object's true appearance never changes between epochs — re-rendering
# PyBullet's camera for every one of the ~10,000 dataset lookups during
# training (500 samples x 2 pairs x 10 epochs) is what was making Step 2
# look "frozen" for minutes with zero visible output. Training now reuses
# these 7 cached renders and only jitters them as tensors (fast, no PyBullet
# calls), while still requiring the CNN to read real RGB pixels.
BASE_CROPS = {}
def build_base_crop_cache():
    for obj in OBJECTS:
        BASE_CROPS[obj["name"]] = get_object_crop(obj, jitter=False)

# ── 8. TOKENIZATION ───────────────────────────────────────────
def tokenize(text):
    words = text.lower().replace("-", " ").split()
    ids = [WORD_TO_ID.get(w, 0) for w in words[:MAX_LEN]]
    ids += [0] * (MAX_LEN - len(ids))
    return torch.tensor(ids, dtype=torch.long)

def build_instruction(obj):
    templates = [
        "pick the {size} {color} {item}",
        "pick the {color} {item}",
        "grab the {size} {color} {item}",
        "get the {color} {size} {item}",
    ]
    return random.choice(templates).format(**obj)

# ── 9. DATASET (real object-specific crops) ───────────────────
class VLADataset(Dataset):
    def __init__(self, n_samples=500):
        self.samples = []
        for _ in range(n_samples):
            obj = random.choice(OBJECTS)
            instr = build_instruction(obj)
            self.samples.append((instr, obj, 1.0))
            other = random.choice([o for o in OBJECTS if o is not obj])
            self.samples.append((instr, other, 0.0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        instr, obj, label = self.samples[idx]
        # Reuse the cached real render for this object, then apply cheap
        # tensor-level jitter (no PyBullet call here -> fast).
        img = BASE_CROPS[obj["name"]].clone()
        if random.random() < 0.9:
            factor = random.uniform(0.85, 1.15)
            img = torch.clamp(img * factor, 0.0, 1.0)
            img = torch.clamp(img + torch.randn_like(img) * 0.02, 0.0, 1.0)
        tokens = tokenize(instr)
        return img, tokens, torch.tensor([label], dtype=torch.float32)

# ── 10. CNN + LANGUAGE MODEL ───────────────────────────────────
class VisualEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(128, 128)

    def forward(self, x):
        x = self.cnn(x).flatten(1)
        return self.fc(x)

class LanguageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(len(VOCAB), 64, padding_idx=0)
        self.gru = nn.GRU(64, 128, batch_first=True)

    def forward(self, tokens):
        x = self.embed(tokens)
        _, h = self.gru(x)
        return h.squeeze(0)

class VLAModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision = VisualEncoder()
        self.language = LanguageEncoder()
        self.fusion = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, image, tokens):
        v = self.vision(image)
        l = self.language(tokens)
        return self.fusion(torch.cat([v, l], dim=1))

model = VLAModel().to(DEVICE)
model_trained = False

def train_model(output_widget):
    global model_trained
    with output_widget:
        print("=" * 60)
        print("STEP 3a: TRAINING CNN + LANGUAGE MODEL")
        print("=" * 60)
        dataset = VLADataset(n_samples=500)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.BCEWithLogitsLoss()

        print(f"{'Epoch':<8} | {'Loss':<10} | {'Accuracy'}")
        print("-" * 40)

        model.train()
        for epoch in range(TRAIN_EPOCHS):
            total_loss, correct, total = 0.0, 0, 0
            for images, tokens, labels in loader:
                images, tokens, labels = images.to(DEVICE), tokens.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                logits = model(images, tokens)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * len(labels)
                preds = (torch.sigmoid(logits) > 0.5).float()
                correct += (preds == labels).sum().item()
                total += len(labels)

            acc = correct / total
            print(f"{epoch+1:02d}/{TRAIN_EPOCHS}   | {total_loss/total:.4f}     | {acc*100:.1f}%")

        model.eval()
        model_trained = True
        print("\nTraining finished.\n")

# ── 11. INFERENCE + FEATURE/SCORE TABLE ───────────────────────
@torch.no_grad()
def score_all_objects(instruction):
    tokens = tokenize(instruction).unsqueeze(0).to(DEVICE)
    results = []
    for obj in OBJECTS:
        img = BASE_CROPS[obj["name"]].clone().unsqueeze(0).to(DEVICE)
        score = torch.sigmoid(model(img, tokens)).item()
        results.append((score, obj))
    results.sort(key=lambda x: x[0], reverse=True)
    return results

def print_score_table(output_widget, instruction, ranking):
    with output_widget:
        print("=" * 60)
        print("STEP 3b: FEATURE -> SCORE TABLE (from trained CNN + GRU)")
        print("=" * 60)
        print(f"Instruction used: \"{instruction}\"\n")
        header = f"{'Score':<8} {'Object':<20} {'Color':<8} {'Shape':<10} {'Size':<8}"
        print(header)
        print("-" * len(header))
        for i, (score, obj) in enumerate(ranking):
            marker = "  <- SELECTED" if i == 0 else ""
            print(f"{score:<8.3f} {obj['name']:<20} {obj['color']:<8} {obj['geom']:<10} {obj['size']:<8}{marker}")
        print()

# ── 12. ROBOT MOTION (STEP 4) ──────────────────────────────────
def get_top_height(obj):
    if obj["geom"] == "box":
        return obj["half"][2]
    if obj["geom"] == "cylinder":
        return obj["height"] / 2
    return obj["radius"]

def move_robot(target, image_widget, steps=40, delay=0.04):
    current = np.array(p.getLinkState(kuka_id, EE_INDEX)[0])
    target = np.array(target)
    for i in range(steps):
        t = i / (steps - 1)
        s = (1 - np.cos(t * np.pi)) / 2
        pos = current + s * (target - current)
        joints = p.calculateInverseKinematics(kuka_id, EE_INDEX, pos.tolist())
        for j in range(NUM_JOINTS):
            p.resetJointState(kuka_id, j, joints[j])
        p.stepSimulation()
        if image_widget is not None:
            image_widget.value = capture_png()
        time.sleep(delay)

def pick_and_carry(obj, output_widget):
    global grasp_constraint
    reset_scene()

    with output_widget:
        print("=" * 60)
        print("STEP 4: ROBOT MOTION (live camera capture)")
        print("=" * 60)

        image_widget = widgets.Image(format="png", width=560, height=400)
        display(image_widget)
        image_widget.value = capture_png()

        pos = np.array(obj["pos"])
        top = get_top_height(obj)
        grasp = pos + [0, 0, top + 0.02]
        above = grasp + [0, 0, 0.13]

        print("Approaching...")
        move_robot(above, image_widget, steps=45)

        print("Reaching...")
        move_robot(grasp, image_widget, steps=30)

        print("Picking...")
        for link in range(-1, NUM_JOINTS):
            p.setCollisionFilterPair(kuka_id, obj["body_id"], link, -1, 0)
        grasp_constraint = p.createConstraint(
            kuka_id, EE_INDEX, obj["body_id"], -1,
            p.JOINT_FIXED, [0, 0, 0], [0, 0, 0], [0, 0, top]
        )
        for _ in range(8):
            p.stepSimulation()
            image_widget.value = capture_png()
            time.sleep(0.04)

        print("Lifting...")
        current = np.array(p.getLinkState(kuka_id, EE_INDEX)[0])
        lift_target = current + [0, 0, 0.20]
        move_robot(lift_target, image_widget, steps=40)

        print(f"\nTask completed! Delivered: {obj['name']}")

# ============================================================
# STEP 1: DISPLAY ALL 7 OBJECTS BEFORE ANYTHING ELSE
# ============================================================
print("=" * 60)
print("STEP 1: SCENE OVERVIEW (7 objects)")
print("=" * 60)

overview_widget = widgets.Image(format="png", width=560, height=400)
overview_widget.value = capture_png()
display(overview_widget)

thumb_boxes = []
shuffled_objects = OBJECTS[:]
random.shuffle(shuffled_objects)
for obj in shuffled_objects:
    thumb = widgets.Image(value=get_object_crop_png(obj), format="png", width=110, height=110)
    label = widgets.Label(value=obj["name"])
    thumb_boxes.append(widgets.VBox([thumb, label], layout=widgets.Layout(align_items="center", margin="4px")))

display(widgets.HBox(thumb_boxes, layout=widgets.Layout(flex_flow="row wrap")))

build_base_crop_cache()
print("Object appearance cache built (7 real renders). Training will reuse these.\n")

# ============================================================
# STEP 2: WIDGET FOR THE USER TO SELECT THE TARGET OBJECT
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: SELECT A TARGET OBJECT")
print("=" * 60)

result_output = widgets.Output()

def on_select(obj, btn):
    for b in select_buttons:
        b.disabled = True
    with result_output:
        clear_output(wait=True)
        print(f"Button clicked: {obj['name']}")
        print("Working... (training runs now, this takes a few seconds)\n")

    try:
        instr = build_instruction(obj)

        # STEP 3: train (live epoch table), then show the feature/score table
        train_model(result_output)
        ranking = score_all_objects(instr)
        print_score_table(result_output, instr, ranking)

        top_score, top_obj = ranking[0]
        with result_output:
            if top_obj["name"] == obj["name"]:
                print(f"Model agrees with your selection: {obj['name']}\n")
            else:
                print(f"Note: model's top pick ({top_obj['name']}) differs from your selection ({obj['name']}).")
                print("Proceeding with YOUR selection.\n")

        # STEP 4: motion, using the object the user actually selected
        pick_and_carry(obj, result_output)
    except Exception as e:
        import traceback
        with result_output:
            print("An error occurred:")
            traceback.print_exc()
    finally:
        for b in select_buttons:
            b.disabled = False

select_buttons = []
for obj in OBJECTS:
    btn = widgets.Button(description=obj["name"], layout=widgets.Layout(width="180px", height="38px"))

    def make_handler(o, b):
        def handler(_b):
            on_select(o, b)
        return handler

    select_buttons.append(btn)

for btn, obj in zip(select_buttons, OBJECTS):
    btn.on_click(make_handler(obj, btn))

display(widgets.VBox([
    widgets.HTML("<b>Click an object to pick it:</b>"),
    widgets.HBox(select_buttons[0:4], layout=widgets.Layout(flex_flow="row wrap")),
    widgets.HBox(select_buttons[4:7], layout=widgets.Layout(flex_flow="row wrap")),
    result_output
]))

print("\nReady. Click a button above to train (live) and then watch the robot move.")

STEP 1: SCENE OVERVIEW (7 objects)


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x020\x00\x00\x01\x90\x08\x02\x00\x00\x00?g\xcdK\x00\…

Object appearance cache built (7 real renders). Training will reuse these.


STEP 2: SELECT A TARGET OBJECT



Ready. Click a button above to train (live) and then watch the robot move.
